# 07 — Dataset Builder
## Construction du dataset final mois par mois

**Processus pour chaque mois** :
1. aggTrades → buckets 1s → features rolling
2. Fusion blockchain (blocs, CDD)
3. Fusion UTXO, mempool, Glassnode, futures
4. Features temporelles
5. Normalisation (rolling z-score causal)
6. Labels (μ, σ, direction à 30s/1m/3m/5m)
7. Validation qualité + test leakage

**Priorités** :
- `priority=1` : aggTrades + blocs (dataset minimal entraînable)
- `priority=2` : + futures, mempool, transactions CDD
- `priority=3` : + UTXO, Glassnode (dataset complet)

In [ ]:
import os, sys, time, json
sys.path.insert(0, "/workspace")

from IPython.display import display
import ipywidgets as widgets

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.config import Config
from btc_pipeline.processors.dataset_builder import build_month, build_full_dataset

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()
config = Config()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Vérification des sources disponibles
# ════════════════════════════════════════════════════════════════════════
print("📋 Sources disponibles sur GCS :")
sources = {
    "aggTrades spot": "raw/binance/spot_aggtrades/",
    "Klines 1m":      "raw/binance/spot_klines_1m/",
    "Futures funding": "raw/binance/futures_funding/",
    "Blockchain blocs": "raw/blockchain/blocks/",
    "Blockchain txs":   "raw/blockchain/transactions/",
    "UTXO snapshots":   "raw/blockchain/utxo_snapshots/",
    "Mempool":          "raw/mempool/",
    "Glassnode":        "raw/onchain_metrics/glassnode/",
    "BGeometrics":      "raw/onchain_metrics/bgeometrics/",
}
for name, prefix in sources.items():
    files = storage.list_files(prefix)
    status = "✅" if files else "❌"
    print(f"  {status} {name}: {len(files)} files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Configuration du build
# ════════════════════════════════════════════════════════════════════════
# ⚠️  MODIFIER CES PARAMÈTRES SELON VOS BESOINS

PRIORITY    = 1       # 1=minimal, 2=intermédiaire, 3=complet
START_YEAR  = 2017
START_MONTH = 8       # Août 2017 (premier mois Binance)
END_YEAR    = 2025    # None = mois dernier complet
END_MONTH   = 2       # None = mois dernier complet

print(f"Configuration :")
print(f"  Priority   : {PRIORITY}")
print(f"  Période    : {START_YEAR}-{START_MONTH:02d} → {END_YEAR}-{END_MONTH:02d}")

## Build complet

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Lancement du build
# ════════════════════════════════════════════════════════════════════════
progress = widgets.IntProgress(
    value=0, max=1,
    description="Build:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)
status_label = widgets.HTML(value="Calcul...")
display(widgets.VBox([progress, status_label]))

def update_progress(current, total, result):
    progress.max = total
    progress.value = current
    status_label.value = (
        f"<b>{result.get('month', '?')}</b> — "
        f"Status: {result.get('status', '?')} — "
        f"{result.get('rows', 0):,} rows — "
        f"{result.get('features', 0)} features — "
        f"{result.get('features_size_mb', 0):.1f} MB"
    )

t_start = time.time()

summary = build_full_dataset(
    storage,
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    priority=PRIORITY,
    config=config,
    progress_callback=update_progress,
)

duration = time.time() - t_start

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Résumé
# ════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("RÉSUMÉ — DATASET BUILDER")
print("═" * 60)
print(f"  Mois traités    : {summary['months_processed']}")
print(f"  Succès          : {summary['success']}")
print(f"  Échoués         : {summary['failed']}")
print(f"  Volume total    : {summary['total_gb']:.2f} GB")
print(f"  Durée           : {duration/3600:.1f}h")
print(f"  Priority        : {summary['priority']}")

# Détail par mois
print("\nDétail par mois :")
for r in summary.get("results", []):
    emoji = "✅" if r["status"] in ("success", "quality_warning") else "❌"
    print(f"  {emoji} {r.get('month', '?')}: {r.get('rows', 0):,} rows, "
          f"{r.get('features', 0)} features, {r.get('status', '?')}")

print("═" * 60)
storage.save_pipeline_state(storage.get_pipeline_state())
print("✅ État sauvegardé")